# Resolución de problemas de búsqueda

En este notebook definiremos un proceso para resolver problemas de búsqueda en los que queremos conocer el camino hasta un estado objetivo. Utilizaremos las torres de Hanói para ilustrarlo.

## Torres de Hanói

### Definición del estado

Para resolver un problema de búsqueda, primero definiremos su estado. En este caso, el estado indica la posición de los discos en las torres. Crearemos una clase que almacene esta información.

Utilizaremos una *dataclass* de Python. Una *dataclass* genera un constructor que inicializa los atributos, lo que permite crear instancias con una sintaxis más sencilla.

In [210]:
from dataclasses import dataclass


@dataclass
class EstatHanoi(object):
    """
    Estat del problema de les torres de Hanoi: guardarem un array amb els discos de cada torre i un punter a l'estat pare.
    Definirem també el mètode __str__ per a poder imprimir l'estat, el hash per a poder utilitzar-lo en un set, i els mètodes __eq__ i __lt__ per a poder comparar dos estats.
    """
    torres: list
    pare: 'EstatHanoi' = None

    def __repr__(self):
        """
        Retorna una cadena de caràcters amb la representació de l'estat.
        """
        out = ""
        for torre in self.torres:
            out += str(torre) + "\n"
        out += "-----\n"
        return out

    def __hash__(self):
        """
        Retorna un hash (nombre enter cqlculat) que representa l'estat.
        """
        return hash(str(self.torres))

    def __eq__(self, altre):
        """
        Retorna True si l'estat és igual a l'estat `altre`. No inclou el pare.
        """
        return self.torres == altre.torres

    def __lt__(self, altre):
        """
        Retorna True si l'estat és menor que l'estat `altre`. No inclou el pare.
        """
        return self.torres < altre.torres


### Definición del problema

Para simplificar el proceso, utilizaremos una clase que permita definir problemas. Así podremos reutilizar el código con problemas distintos.

In [211]:
class Problema(object):
    """Aquesta és la classe abstracta per a un problema formal. Per a resoldre un problema nou crearem una classe heretada, sobrescrivint `accions` i `accio`, i potser altres mètodes.
    Per defecte no hi ha definida cap heurística (h torna sempre 0), i tots els costos d'acció són 1.
"""

    def __init__(self, inicial=None, final=None, **kwds):
        """El constructor especifica l'estat inicial i final (si es coneix) i pot contenir altres arguments."""
        self.__dict__.update(inicial=inicial, final=final, **kwds)

    def accions(self, estat):
        """Retorna les accions possibles que es poden fer en l'estat `estat`."""
        raise NotImplementedError

    def accio(self, estat, action):
        """Retorna l'estat que resulta de fer l'acció `accio` en l'estat `estat`."""
        raise NotImplementedError

    def es_resultat(self, estat):
        """
        Retorna True si l'estat `estat` és un estat objectiu. Per defecte, compara amb l'estat final, si s'ha especificat.
        """
        return estat == self.final

    def cost_accio(self, estat, a, estat1):
        """
        Retorna el cost de fer l'acció `accio` en l'estat `estat` fins a arribar a l'estat `estat1`. Per defecte, retorna 1.
        """
        return 1

    def h(self, estat):
        """
        Retorna un valor heurístic per a l'estat `estat`. Per defecte, retorna 0.
        Per ser admissible, aquest valor heurístic no pot sobreestimar el cost per arribar a l'estat objectiu.
        """
        return 0

    def __str__(self):
        return '{}({!r}, {!r})'.format(
            type(self).__name__, self.inicial, self.final)

Este código define una clase base llamada `Problema` para resolver problemas de búsqueda en Python. Es una clase abstracta que establece la interfaz de un problema de búsqueda genérico. Sus métodos pueden sobrescribirse para adaptarlos a un problema concreto.

Los métodos de `Problema` son:

- `__init__(self, inicial=None, final=None, **kwds)`: constructor de la clase. Recibe `inicial` y `final`, que representan los estados inicial y final. También admite argumentos adicionales con nombre para personalizar la instancia.
- `accions(self, estat)`: debe sobrescribirse para devolver las acciones posibles desde un estado.
- `accio(self, estat, action)`: debe sobrescribirse para devolver el estado resultante de aplicar una acción.
- `es_resultat(self, estat)`: devuelve `True` si el estado es final. Por defecto, lo compara con el estado final indicado en el constructor.
- `cost_accio(self, estat, a, estat1)`: devuelve el coste de aplicar una acción para llegar a otro estado. Por defecto, el coste es 1.
- `h(self, estat)`: devuelve el valor heurístico de un estado. Por defecto, es 0; se puede sobrescribir para definir una heurística específica.
- `__str__(self)`: devuelve una representación textual de la instancia.

Resolveremos el problema de las torres de Hanói. Para ello, crearemos una clase que herede de `Problema` y sobrescribiremos los métodos necesarios.

In [212]:
class ProblemaHanoi(Problema):
    """
    Problema de les torres de Hanoi: donada una configuració inicial, trobar una seqüència d'accions per a arribar a una configuració final.
    """

    def accions(self, estat):
        """
        Retorna les accions possibles en un estat donat. Les accions són parells de nombres que representen les torres d'origen i destí.
        """
        estats = []
        for i in range(3):
            for j in range(3):
                if i != j and self.es_valida(estat, i, j):
                    estats.append((i, j))
        return estats

    def accio(self, estat, accio):
        """
        Retorna l'estat que resulta de fer una acció en un estat donat. 
        """
        i, j = accio
        nou_estat = EstatHanoi([list(torre) for torre in estat.torres], pare=estat)
        nou_estat.torres[j].append(nou_estat.torres[i].pop())
        return nou_estat

    def es_valida(self, estat, i, j):
        """
        Retorna True si es pot moure un disc de la torre i a la torre j.
        Es pot moure un disc de la torre i a la torre j si la torre i no està buida i la torre j està buida o el disc de la torre i és més petit que el disc de la torre j.
        """
        if len(estat.torres[i]) == 0:
            return False
        if len(estat.torres[j]) == 0:
            return True
        return estat.torres[i][-1] < estat.torres[j][-1]

    def es_resultat(self, estat):
        """
        Retorna True si l'estat és un estat final.
        """
        return estat == self.final

    def __str__(self):
        return '{}({!r}, {!r})'.format(
            type(self).__name__, self.inicial, self.final)

### Definición de los estados inicial y final y creación del problema

Para resolver el problema, definiremos los estados inicial y final. En el estado inicial, todos los discos están en la primera torre y las otras dos están vacías. En el estado final, todos están en la última torre y las otras dos están vacías.

A continuación, crearemos una instancia del problema con estos estados.

In [213]:
# Exemple de problema de les torres de Hanoi
estat_inicial = EstatHanoi([[5, 4, 3, 2, 1], [], []])
estat_final = EstatHanoi([[], [], [5, 4, 3, 2, 1]])

p = ProblemaHanoi(estat_inicial, estat_final)
print(p)

ProblemaHanoi([5, 4, 3, 2, 1]
[]
[]
-----
, []
[]
[5, 4, 3, 2, 1]
-----
)


### Implementación de la búsqueda de coste uniforme

A continuación, implementaremos la búsqueda de coste uniforme. El algoritmo expande los nodos de menor coste hasta encontrar un estado objetivo. Para ello utilizaremos una cola de prioridad, que permite añadir estados pendientes y extraer el de menor coste.

Podríamos utilizar búsqueda en anchura, pero la búsqueda de coste uniforme es más general.

In [214]:
import heapq


def cerca_cost_uniforme(problema):
    """
    Cerca de cost uniforme: expandeix l'estat amb menor cost fins a trobar un node objectiu.
    """

    # Inicialitzem la frontera amb l'estat inicial
    inicial = problema.inicial

    # La frontera és una cua de prioritat on el cost del node és la prioritat de la cua. El cost del node és el cost del camí per arribar al node (inicialment 0).
    frontera = [(0, inicial)]

    # Inicialitzem el conjunt de nodes visitats
    visitats = set()

    # Mentre la frontera no estigui buida
    while frontera:
        # Extreiem el node amb menor cost de la frontera
        cost, estat = heapq.heappop(frontera)
        # Afegim l'estat del node als visitats
        visitats.add(estat)
        # Impimim l'estat
        print(estat)

        # Si l'estat és un estat final, retornem el node
        if problema.es_resultat(estat):
            # Imprimim el nombre de nodes visitats
            print(f"Nombre de nodes visitats: {len(visitats)}")

            # Retornem l'estat
            return estat

        # Per a cada acció possible en l'estat
        for accio in problema.accions(estat):
            # Calculem el nou estat
            nou_estat = problema.accio(estat, accio)
            print(nou_estat)
            # Si el nou estat no està visitat
            if nou_estat not in visitats:
                # Afegim el nou estat a la frontera amb el cost del camí acumulat
                cost_accio = problema.cost_accio(estat, accio, nou_estat)
                heapq.heappush(
                    frontera,
                    (cost + cost_accio, nou_estat)
                )

In [215]:
# Exemple de cerca de cost uniforme
desti = cerca_cost_uniforme(p)
desti

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3]
[2]
[1]
-----

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3, 1]
[]
[2]
-----

[5, 4, 3]
[]
[2, 1]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3]
[2]
[1]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3]
[2, 1]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3]
[]
[2, 1]
-----

[5, 4]
[3]
[2, 1]
-----

[5, 4, 3, 1]
[]
[2]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3]
[2, 1]
[]
-----

[5, 4]
[2, 1]
[3]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3]
[2]
[1]
-----

[5, 4, 3, 1]
[]
[2]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3]
[]
[2, 1]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3]
[2, 1]
[]
-----

[5, 4, 3]
[2]

[]
[]
[5, 4, 3, 2, 1]
-----

### Recuperación de la ruta

Para recuperar la ruta, iremos desde el nodo objetivo hasta el inicial, siguiendo la información del atributo `pare` de cada nodo.

A continuación definiremos una función para reconstruirla.

In [216]:
def recuperar_cami(estat):
    """
    Retorna el camí des de l'estat `estat` fins a l'estat inicial.
    """
    cami = []
    while estat:
        cami.append(estat)
        estat = estat.pare
    return cami[::-1]

In [217]:
# Exemple de recuperació del camí
recuperar_cami(desti)

[[5, 4, 3, 2, 1]
 []
 []
 -----,
 [5, 4, 3, 2]
 []
 [1]
 -----,
 [5, 4, 3]
 [2]
 [1]
 -----,
 [5, 4, 3]
 [2, 1]
 []
 -----,
 [5, 4]
 [2, 1]
 [3]
 -----,
 [5, 4, 1]
 [2]
 [3]
 -----,
 [5, 4, 1]
 []
 [3, 2]
 -----,
 [5, 4]
 []
 [3, 2, 1]
 -----,
 [5]
 [4]
 [3, 2, 1]
 -----,
 [5]
 [4, 1]
 [3, 2]
 -----,
 [5, 2]
 [4, 1]
 [3]
 -----,
 [5, 2, 1]
 [4]
 [3]
 -----,
 [5, 2, 1]
 [4, 3]
 []
 -----,
 [5, 2]
 [4, 3]
 [1]
 -----,
 [5]
 [4, 3, 2]
 [1]
 -----,
 [5]
 [4, 3, 2, 1]
 []
 -----,
 []
 [4, 3, 2, 1]
 [5]
 -----,
 [1]
 [4, 3, 2]
 [5]
 -----,
 [1]
 [4, 3]
 [5, 2]
 -----,
 []
 [4, 3]
 [5, 2, 1]
 -----,
 [3]
 [4]
 [5, 2, 1]
 -----,
 [3]
 [4, 1]
 [5, 2]
 -----,
 [3, 2]
 [4, 1]
 [5]
 -----,
 [3, 2, 1]
 [4]
 [5]
 -----,
 [3, 2, 1]
 []
 [5, 4]
 -----,
 [3, 2]
 []
 [5, 4, 1]
 -----,
 [3]
 [2]
 [5, 4, 1]
 -----,
 [3]
 [2, 1]
 [5, 4]
 -----,
 []
 [2, 1]
 [5, 4, 3]
 -----,
 [1]
 [2]
 [5, 4, 3]
 -----,
 [1]
 []
 [5, 4, 3, 2]
 -----,
 []
 []
 [5, 4, 3, 2, 1]
 -----]

### Uso de una heurística

Para evitar expandir estados que no nos acercan al objetivo, utilizaremos una función heurística que estime el coste restante. Debe ser admisible, es decir, no puede sobreestimar el coste hasta el objetivo.

Para las torres de Hanói, una heurística admisible es el número de discos que faltan por mover. Cada acción mueve un disco y cuesta 1, así que el coste restante siempre será igual o superior a ese número.

Implementaremos `ProblemaHanoiHeuristic`, que hereda de `ProblemaHanoi` y sobrescribe `h` para devolver el número de discos que faltan por mover.

In [218]:
class ProblemaHanoiHeuristic(ProblemaHanoi):
    """
    Problema de les torres de Hanoi: donada una configuració inicial, trobar una seqüència d'accions per a arribar a una configuració final.
    """

    def h(self, estat):
        """
        Retorna el nombre de discos que falten per moure.
        """
        return len(estat.torres[0] + estat.torres[1])

### Implementación del algoritmo A*

Para utilizar la heurística, implementaremos A*. Este algoritmo expande los nodos con menor coste acumulado más heurística hasta encontrar un estado objetivo.

In [219]:
def a_estrella(problema):
    """
    Cerca A*: expandeix l'estat amb menor cost + heurística fins a trobar un node objectiu.
    """

    # Inicialitzem la frontera amb l'estat inicial
    inicial = problema.inicial

    # La frontera és una cua de prioritat on el cost del node és la prioritat de la cua. El cost del node és el cost del camí per arribar al node (inicialment 0).
    frontera = [(0, inicial)]

    # Inicialitzem el conjunt de nodes visitats
    visitats = set()

    # Mentre la frontera no estigui buida
    while frontera:
        # Extreiem el node amb menor cost de la frontera
        cost, estat = heapq.heappop(frontera)
        # Afegim l'estat del node als visitats
        visitats.add(estat)
        # Impimim l'estat
        print(estat)

        # Si l'estat és un estat final, retornem el node
        if problema.es_resultat(estat):
            # Imprimim el nombre de nodes visitats
            print(f"Nombre de nodes visitats: {len(visitats)}")

            # Retornem l'estat
            return estat

        # Per a cada acció possible en l'estat
        for accio in problema.accions(estat):
            # Calculem el nou estat
            nou_estat = problema.accio(estat, accio)
            print(nou_estat)
            # Si el nou estat no està visitat
            if nou_estat not in visitats:
                # Afegim el nou estat a la frontera amb el cost del camí acumulat
                cost_accio = problema.cost_accio(estat, accio, nou_estat)
                heapq.heappush(
                    frontera,
                    (cost + cost_accio + problema.h(nou_estat), nou_estat)
                )

In [220]:
# Exemple de cerca A*
p = ProblemaHanoiHeuristic(estat_inicial, estat_final)
desti = a_estrella(p)

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3]
[2]
[1]
-----

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3]
[2]
[1]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3]
[2, 1]
[]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3, 1]
[]
[2]
-----

[5, 4, 3]
[]
[2, 1]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3, 2]
[1]
[]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3, 2, 1]
[]
[]
-----

[5, 4, 3, 2]
[]
[1]
-----

[5, 4, 3]
[]
[2, 1]
-----

[5, 4]
[3]
[2, 1]
-----

[5, 4, 3, 1]
[]
[2]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3]
[2, 1]
[]
-----

[5, 4]
[2, 1]
[3]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3]
[2]
[1]
-----

[5, 4, 3, 1]
[]
[2]
-----

[5, 4, 3]
[1]
[2]
-----

[5, 4, 3]
[]
[2, 1]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3, 1]
[2]
[]
-----

[5, 4, 3]
[2, 1]
[]
-----

[5, 4, 3]
[2]

## Ruta por Castellón

Veremos otro ejemplo de búsqueda con un problema de rutas por Castellón. Buscaremos el camino más corto desde la UJI hasta otro punto de la ciudad mediante un mapa simplificado con las distancias entre los distintos lugares.

### Definición del estado

Para resolver el problema, primero definiremos su estado. En este caso, representa la posición del viajero en el mapa. Crearemos una clase para almacenar esta información.

In [221]:
from dataclasses import dataclass


@dataclass
class EstatCastello(object):
    """
    Estat del problema del viatjer per Castelló: guardarem un enter amb la posició del viatjer i un punter a l'estat pare.
    Definirem també el mètode __str__ per a poder imprimir l'estat, el hash per a poder utilitzar-lo en un set, i els mètodes __eq__ i __lt__ per a poder comparar dos estats.
    """
    posicio: int
    pare: 'EstatCastello' = None

    def __repr__(self):
        """
        Retorna una cadena de caràcters amb la representació de l'estat.
        """
        return str(self.posicio)

    def __hash__(self):
        """
        Retorna un hash (nombre enter cqlculat) que representa l'estat.
        """
        return hash(self.posicio)

    def __eq__(self, altre):
        """
        Retorna True si l'estat és igual a l'estat `altre`. No inclou el pare.
        """
        return self.posicio == altre.posicio

    def __lt__(self, altre):
        """
        Retorna True si l'estat és menor que l'estat `altre`. No inclou el pare.
        """
        return self.posicio < altre.posicio

## Definición del problema

A continuación, crearemos una clase que herede de `Problema` y sobrescribiremos los métodos necesarios.

In [222]:
class ProblemaCastello(Problema):
    """
    Problema del viatjer per Castelló: donada una posició inicial, trobar una seqüència d'accions per a arribar a una posició final.
    """

    def accions(self, estat):
        """
        Retorna les accions possibles en un estat donat. Les accions són parells de nombres que representen les posicions d'origen i destí.
        Utilitzarem una llista amb les distàncies reals precalculades per a cada punt.
        """
        estats = []
        for i in range(len(self.distancies)):
            if self.distancies[estat.posicio][i] > 0:
                estats.append((estat.posicio, i))
        return estats

    def accio(self, estat, accio):
        """
        Retorna l'estat que resulta de fer una acció en un estat donat. 
        """
        nou_estat = EstatCastello(accio[1], pare=estat)
        return nou_estat

    def es_resultat(self, estat):
        """
        Retorna True si l'estat és un estat final.
        """
        return estat == self.final

    def __str__(self):
        return '{}({!r}, {!r})'.format(
            type(self).__name__, self.inicial, self.final)

## Definición de los estados inicial y final y creación del problema

El estado inicial es la ubicación de la UJI y el estado final es el punto de Castellón al que queremos llegar; en este ejemplo, el parque Ribalta.

A continuación, crearemos una instancia del problema con estos estados.

In [223]:
# Exemple de problema del viatjer per Castelló
distancies = [
    [0, 2, 3, 5, 6, 5, 6, 7, 8, 9],  # UJI
    [2, 0, 1, 3, 4, 3, 4, 5, 6, 7],  # Plaça Borrull
    [3, 1, 0, 2, 3, 2, 3, 4, 5, 6],  # Plaça Maria Agustina
    [5, 3, 2, 0, 1, 2, 3, 4, 5, 6],  # Plaça Santa Clara
    [6, 4, 3, 1, 0, 1, 2, 3, 4, 5],  # Plaça Major
    [5, 3, 2, 2, 1, 0, 1, 2, 3, 4],  # Plaça Hort dels Corders
    [6, 4, 3, 3, 2, 1, 0, 1, 2, 3],  # Plaça Huerto Sogueros
    [7, 5, 4, 4, 3, 2, 1, 0, 1, 2],  # Plaça Fadrell
    [8, 6, 5, 5, 4, 3, 2, 1, 0, 1],  # Plaça Tetuán
    [9, 7, 6, 6, 5, 4, 3, 2, 1, 0],  # Parc Ribalta

]
estat_inicial = EstatCastello(0)
estat_final = EstatCastello(9)

p = ProblemaCastello(estat_inicial, estat_final, distancies=distancies)

print(p)

ProblemaCastello(0, 9)


## Implementación de la búsqueda de coste uniforme

A continuación, aplicaremos la búsqueda de coste uniforme.

In [224]:
desti = cerca_cost_uniforme(p)
desti

0
1
2
3
4
5
6
7
8
9
1
0
2
3
4
5
6
7
8
9
2
0
1
3
4
5
6
7
8
9
3
0
1
2
4
5
6
7
8
9
4
0
1
2
3
5
6
7
8
9
5
0
1
2
3
4
6
7
8
9
6
0
1
2
3
4
5
7
8
9
7
0
1
2
3
4
5
6
8
9
8
0
1
2
3
4
5
6
7
9
9
Nombre de nodes visitats: 10


9

In [225]:
recuperar_cami(desti)

[0, 9]

## Uso de una heurística

Para las rutas por Castellón, una heurística admisible es la distancia Manhattan hasta el destino. Suponiendo que cada acción acerca al viajero al destino y cuesta 1, el coste restante será igual o superior a esa distancia.

Implementaremos `ProblemaCastelloHeuristic`, que hereda de `ProblemaCastello` y sobrescribe `h` para devolver la distancia en línea recta hasta el destino.

In [226]:
class ProblemaCastelloHeuristic(ProblemaCastello):
    """
    Problema del viatjer per Castelló: donada una posició inicial, trobar una seqüència d'accions per a arribar a una posició final.
    """

    def h(self, estat):
        """
        Retorna la distància Manhattan fins al punt final.
        """
        return abs(estat.posicio - self.final.posicio)

In [227]:
# Exemple de cerca A*
p = ProblemaCastelloHeuristic(estat_inicial, estat_final, distancies=distancies)
desti = a_estrella(p)
recuperar_cami(desti)

0
1
2
3
4
5
6
7
8
9
9
Número de nodos visitados: 2


[0, 9]